# Quick Start Guide - Azure AI Foundry

This notebook provides a hands-on introduction to Azure AI Foundry. You'll learn how to:
1. Initialize the AI Project client
2. List available models
3. Create a simple chat completion request
4. Create a basic AI agent
5. Handle basic error scenarios

## Prerequisites
- Completed environment setup from previous notebook
- Azure credentials configured

## Import Required Libraries and Setup

In the next cell, we'll:
1. Import the necessary Azure SDK libraries for authentication and AI Projects
2. Import standard Python libraries for environment variables and JSON handling
3. Initialize Azure credentials using DefaultAzureCredential
   - This will automatically use your logged-in Azure CLI credentials
   - Alternatively, it can use other authentication methods like environment variables or managed identity


In [ ]:
# Import required libraries
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
import os
import json

# Initialize credentials
credential = DefaultAzureCredential()

## Initialize AI Project Client

> **Note:** Before proceeding, ensure you:
> 1. Copy your `.env.example` file to `.env` from the root directory
> 2. Set `PROJECT_ENDPOINT` in your `.env` file
> 3. Have a Project already provisioned in Microsoft Foundry

You can find your **Project endpoint** on the [Microsoft Foundry](https://ai.azure.com) Home page (`https://<resource>.services.ai.azure.com/api/projects/<project>`).


## Creating the AI Project Client

In the next cell, we'll create an AI Project client using your project endpoint from the `.env` file.
> **Note:** This example uses the synchronous client. For higher performance scenarios, you can also create an asynchronous client by importing `asyncio` and using the async methods from `AIProjectClient`.

The client will be used to:
- Connect to your Azure AI Project using your project endpoint from the `.env` file
- Authenticate using Azure credentials
- Enable making inference requests to your deployed models


In [ ]:
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

try:
    # OpenAI client used for chat/embeddings/responses.
    client = AIProjectClient(
        endpoint=os.getenv("PROJECT_ENDPOINT"),
        credential=credential,
    )
    openai_client = client.get_openai_client()
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")

## Create a Simple Completion
Let's try a basic completion request.

Now that we have an authenticated client, we'll use the **OpenAI client** from `client.get_openai_client()` to make a chat completion. The code below:
1. Reads `MODEL_DEPLOYMENT_NAME` from our `.env` file
2. Calls `openai_client.chat.completions.create(...)`

Using the deployment name makes it easy to switch models without changing code. This works for any model deployed in your Foundry project (Azure OpenAI, Microsoft, or other providers).


In [ ]:
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

try:
    # Chat completion via the OpenAI client (works for any deployed model by name)
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[{"role": "user", "content": "How to be healthy in one sentence?"}],
    )
    print(response.choices[0].message.content)
except Exception as e:
    print(f"An error occurred: {str(e)}")

## Create a simple Agent

Using AI Agent Service, we can create a simple agent to answer health related questions.

Let's explore Azure AI Agent Service, a powerful tool for building intelligent agents.

Azure AI Agent Service is a fully managed service that helps developers build, deploy, and scale AI agents
without managing infrastructure. It combines large language models with tools that allow agents to:
- Answer questions using RAG (Retrieval Augmented Generation)
- Perform actions through tool calling 
- Automate complex workflows

The code below demonstrates how to:
1. Create an agent with a code interpreter tool
2. Create a conversation thread
3. Send a message requesting BMI analysis 
4. Process the request and get results
5. Save any generated visualizations

The agent will use the model specified in our .env file (MODEL_DEPLOYMENT_NAME) and will have access
to a code interpreter tool for creating visualizations. This showcases how agents can combine
natural language understanding with computational capabilities.

> The visualization will be saved as a PNG file in the same folder as this notebook.
 



In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool

try:
    # Create an agent version with the Code Interpreter tool
    agent = client.agents.create_version(
        agent_name="bmi-calculator",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=(
                "You are a health analyst who calculates BMI using US metrics (pounds, feet/inches). "
                "Use average US female measurements: 5'4\" (64 inches) and 130 pounds. "
                "Create a visualization showing where this BMI falls on the scale and save it as a PNG file."
            ),
            tools=[CodeInterpreterTool()],
        ),
        description="BMI calculator agent with the code interpreter tool.",
    )

    # Create a conversation and run the agent via the Responses API
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=(
            "Calculate BMI for an average US female (5'4\", 130 lbs). "
            "Create a visualization showing where this BMI falls on the standard BMI scale from 15 to 35. "
            "Include the standard BMI categories (Underweight, Normal, Overweight, Obese) and save it as a PNG file."
        ),
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

    # Print the analysis text from the assistant
    print(f"Analysis: {response.output_text}")

    # Save any generated visualization (container_file_citation annotations)
    for item in (response.output or []):
        if getattr(item, "type", "") != "message":
            continue
        for content in (getattr(item, "content", None) or []):
            for ann in (getattr(content, "annotations", None) or []):
                if getattr(ann, "type", "") == "container_file_citation":
                    file_name = os.path.basename(ann.filename) or f"bmi_analysis_{ann.file_id}.png"
                    file_content = openai_client.containers.files.content.retrieve(
                        file_id=ann.file_id, container_id=ann.container_id
                    )
                    with open(file_name, "wb") as f:
                        f.write(file_content.read())
                    print(f"Analysis saved as: {file_name}")

except Exception as e:
    print(f"An error occurred: {str(e)}")

### 👀 See your agent live in the Foundry portal

Before deleting the agent, confirm it now exists in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`bmi-calculator`** in the list — the agent you just created from code.
4. Return to this notebook and run the next cell to delete the agent.

In [ ]:
# Cleanup by deleting the agent version
client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print(f"Deleted agent: {agent.name}")